In [3]:
import numpy as np
import pandas as pd

zones = pd.read_csv("zones.csv")
weather = pd.read_csv("weather_logs.csv")
cost = pd.read_csv("delivery_cost.csv")

print("Zones.shape:" , zones.shape)
print("Weather.shape:" , weather.shape)
print("Cost.shape:" , cost.shape)

Zones.shape: (20, 5)
Weather.shape: (1815, 6)
Cost.shape: (1810, 6)


In [4]:
print("Missing values in weather:\n" , weather.isnull().sum())
print("\nMissing values in cost:\n" , cost.isnull().sum())

Missing values in weather:
 log_id               0
zone_id              0
date                 0
wind_speed_kmph     54
visibility_km       18
precipitation_mm     0
dtype: int64

Missing values in cost:
 record_id             0
zone_id               0
date                  0
deliveries_count      0
cost_per_delivery    36
battery_used_pct      0
dtype: int64


In [5]:
for name in zones["zone_name"]:
    print(repr(name))

'Gachibowli'
'kondapur'
'  Madhapur  '
'Hitech City'
'Jubilee Hills'
'banjara hills'
'Secunderabad'
'Uppal'
'LB Nagar'
'Dilsukhnagar'
'Kukatpally'
'Miyapur'
'Ameerpet'
'Begumpet'
'Malakpet'
'Charminar'
'Attapur'
'manikonda'
'Shamshabad'
'Nacharam'


In [6]:
zones["zone_name"] = zones["zone_name"].str.strip().str.title()
for name in zones["zone_name"]:
    print(repr(name))

'Gachibowli'
'Kondapur'
'Madhapur'
'Hitech City'
'Jubilee Hills'
'Banjara Hills'
'Secunderabad'
'Uppal'
'Lb Nagar'
'Dilsukhnagar'
'Kukatpally'
'Miyapur'
'Ameerpet'
'Begumpet'
'Malakpet'
'Charminar'
'Attapur'
'Manikonda'
'Shamshabad'
'Nacharam'


In [7]:
zones["zone_name"] = zones["zone_name"].replace("Lb Nagar", "LB Nagar")
for name in zones["zone_name"]:
    print(repr(name))


'Gachibowli'
'Kondapur'
'Madhapur'
'Hitech City'
'Jubilee Hills'
'Banjara Hills'
'Secunderabad'
'Uppal'
'LB Nagar'
'Dilsukhnagar'
'Kukatpally'
'Miyapur'
'Ameerpet'
'Begumpet'
'Malakpet'
'Charminar'
'Attapur'
'Manikonda'
'Shamshabad'
'Nacharam'


In [8]:
# Fill missing wind speed with that zone's own average wind speed
weather["wind_speed_kmph"] = weather.groupby("zone_id")["wind_speed_kmph"].transform(
    lambda x: x.fillna(x.mean())
)

# Same approach for visibility
weather["visibility_km"] = weather.groupby("zone_id")["visibility_km"].transform(
    lambda x: x.fillna(x.mean())
)

# Confirm no missing values remain
print(weather.isna().sum())

log_id              0
zone_id             0
date                0
wind_speed_kmph     0
visibility_km       0
precipitation_mm    0
dtype: int64


In [9]:
# Look at the highest wind speed values to spot anything impossible
weather["wind_speed_kmph"].sort_values(ascending=False).head(10)

1103    330.564014
1023    290.866253
1371    281.536134
1655    226.494143
325     225.747040
758     210.151436
1033     62.000000
1359     57.800000
1770     50.700000
1334     50.100000
Name: wind_speed_kmph, dtype: float64

In [10]:
# some of these wind speed values are way too high to be real (like 300+ km/h)
# treating anything above 100 as a bad sensor reading and marking it as missing for now
weather.loc[weather["wind_speed_kmph"] > 100, "wind_speed_kmph"] = np.nan

# now fill those in using each zone's own average wind speed, same as I did earlier for the actual missing values
weather["wind_speed_kmph"] = weather.groupby("zone_id")["wind_speed_kmph"].transform(
    lambda x: x.fillna(x.mean())
)

In [11]:
# checking the top values again to make sure the crazy outliers are gone
weather["wind_speed_kmph"].sort_values(ascending=False).head(10)

1033    62.0
1359    57.8
1770    50.7
1334    50.1
1620    48.9
1686    47.6
189     46.1
988     44.8
366     44.4
320     42.6
Name: wind_speed_kmph, dtype: float64

In [12]:
# checking for any negative delivery costs - shouldn't be possible, so these are bad data
cost[cost["cost_per_delivery"] < 0]

,record_id,zone_id,date,deliveries_count,cost_per_delivery,battery_used_pct
220,221,3,2025-06-10,7,-64.64,59.8
442,443,5,2025-07-22,10,-34.50,28.4
540,541,7,2025-05-01,15,-32.60,17.5
882,883,10,2025-07-12,7,-64.15,55.1
1024,1025,12,2025-06-04,3,-79.07,82.3


In [13]:
# these negative costs are probably just a sign error - the actual number looks fine
# so instead of treating them as missing, just flip them back to positive
cost.loc[cost["cost_per_delivery"] < 0, "cost_per_delivery"] = cost["cost_per_delivery"].abs()

In [14]:
cost[cost["cost_per_delivery"] < 0]

,record_id,zone_id,date,deliveries_count,cost_per_delivery,battery_used_pct


In [15]:
# some delivery costs are missing, filling them in using each zone's own average cost
# same trick i used for the wind speed missing values earlier
cost["cost_per_delivery"] = cost.groupby("zone_id")["cost_per_delivery"].transform(
    lambda x: x.fillna(x.mean())
)

In [16]:
# battery percentage can't go above 100, so anything over that is a bad reading
# marking those as missing first, same as i did for the crazy wind speed numbers
cost.loc[cost["battery_used_pct"] > 100, "battery_used_pct"] = np.nan

# now filling those in using each zone's normal battery usage average
cost["battery_used_pct"] = cost.groupby("zone_id")["battery_used_pct"].transform(
    lambda x: x.fillna(x.mean())
)

In [17]:
# last check - making sure nothing is missing anymore and no weird values are left
print(cost.isna().sum())
print("Max battery %:", cost["battery_used_pct"].max())
print("Min cost:", cost["cost_per_delivery"].min())

record_id            0
zone_id              0
date                 0
deliveries_count     0
cost_per_delivery    0
battery_used_pct     0
dtype: int64
Max battery %: 87.1
Min cost: 23.79


In [18]:
# checking how many exact duplicate rows exist in each table before removing them
print("Duplicate rows in weather:", weather.duplicated().sum())
print("Duplicate rows in cost:", cost.duplicated().sum())

Duplicate rows in weather: 15
Duplicate rows in cost: 10


In [19]:
# dropping the duplicate rows, just keeping one copy of each
weather = weather.drop_duplicates()
cost = cost.drop_duplicates()

# checking again to make sure the duplicates are actually gone now
print("Duplicate rows in weather:", weather.duplicated().sum())
print("Duplicate rows in cost:", cost.duplicated().sum())

# also checking row counts went back to normal after dropping them
print("Weather shape:", weather.shape)
print("Cost shape:", cost.shape)

Duplicate rows in weather: 0
Duplicate rows in cost: 0
Weather shape: (1800, 6)
Cost shape: (1800, 6)


In [20]:
# saving the cleaned versions so i don't lose all this work
# these will overwrite the old messy csv files with the clean ones
zones.to_csv("zones_clean.csv", index=False)
weather.to_csv("weather_clean.csv", index=False)
cost.to_csv("delivery_cost_clean.csv", index=False)

print("saved all 3 cleaned files")

saved all 3 cleaned files


In [21]:
# rebuilding risk scores from the cleaned data now
# merging weather with zones so i have obstacle_density available alongside daily weather
merged = weather.merge(zones[["zone_id", "obstacle_density"]], on="zone_id")

# turning each factor into a 0-1 "danger" score before combining them
wind_danger = merged["wind_speed_kmph"] / merged["wind_speed_kmph"].max()
visibility_danger = 1 - (merged["visibility_km"] / 10)
obstacle_danger = merged["obstacle_density"] / 10

# combining with weights - wind matters most, then visibility, then obstacles
danger_score = (wind_danger * 0.4) + (visibility_danger * 0.3) + (obstacle_danger * 0.3)

# flipping it so 100 = safest, 0 = most dangerous - easier to read this way
safety_score = round((1 - danger_score) * 100, 1)

risk_scores = pd.DataFrame({
    "zone_id": merged["zone_id"],
    "date": merged["date"],
    "safety_score": safety_score
})

risk_scores.to_csv("risk_scores_clean.csv", index=False)
print(risk_scores.head())

   zone_id        date  safety_score
0        1  2025-05-01          76.3
1        1  2025-05-02          79.6
2        1  2025-05-03          70.0
3        1  2025-05-04          83.6
4        1  2025-05-05          69.8


In [22]:
# now putting all 4 clean tables into one sqlite database, like i did before but with the clean data this time
import sqlite3

conn = sqlite3.connect("drone_delivery_clean.db")

zones.to_sql("zones", conn, if_exists="replace", index=False)
weather.to_sql("weather_logs", conn, if_exists="replace", index=False)
cost.to_sql("delivery_cost", conn, if_exists="replace", index=False)
risk_scores.to_sql("risk_scores", conn, if_exists="replace", index=False)

print("database built with clean data")
conn.close()

database built with clean data


In [23]:
# checking exactly where this notebook is running from, and confirming the db file is actually there
import os
print(os.getcwd())
print(os.listdir())

C:\Users\23831.DESKTOP-NJ1PP00\Favorites\Links
['.ipynb_checkpoints', 'Data_cleaning.ipynb', 'delivery_cost.csv', 'delivery_cost_clean.csv', 'desktop.ini', 'drone_delivery_clean.db', 'queries.sql', 'risk_scores_clean.csv', 'weather_clean.csv', 'weather_logs.csv', 'zones.csv', 'zones_clean.csv']


In [24]:
# running the rollout priority query directly and saving the result as a csv
import sqlite3

conn = sqlite3.connect("drone_delivery_clean.db")

query1 = """
SELECT 
    z.zone_name,
    ROUND(AVG(r.safety_score), 1) AS avg_safety,
    ROUND(AVG(c.cost_per_delivery), 1) AS avg_cost,
    ROUND(
        (AVG(r.safety_score) * 0.6) + 
        ((100 - AVG(c.cost_per_delivery)) * 0.4), 
    1) AS rollout_priority_score
FROM zones z
JOIN risk_scores r ON z.zone_id = r.zone_id
JOIN delivery_cost c ON z.zone_id = c.zone_id AND r.date = c.date
GROUP BY z.zone_name
ORDER BY rollout_priority_score DESC;
"""

rollout_priority = pd.read_sql(query1, conn)
rollout_priority.to_csv("rollout_priority.csv", index=False)
print(rollout_priority)

        zone_name  avg_safety  avg_cost  rollout_priority_score
0      Kukatpally        79.2      30.7                    75.3
1       Charminar        80.9      37.9                    73.4
2   Banjara Hills        80.9      37.9                    73.4
3        Begumpet        81.5      40.0                    72.9
4    Secunderabad        74.8      33.0                    71.7
5      Gachibowli        78.5      48.6                    67.7
6        Malakpet        70.7      38.7                    66.9
7        Nacharam        73.9      44.0                    66.8
8         Attapur        74.1      44.6                    66.6
9     Hitech City        82.9      59.9                    65.8
10  Jubilee Hills        63.8      37.9                    63.1
11      Manikonda        74.6      55.5                    62.6
12       Ameerpet        83.7      71.2                    61.8
13          Uppal        84.0      72.7                    61.4
14       LB Nagar        73.3      59.6 

In [25]:
# running the risky days query and saving the result as a csv
query2 = """
SELECT 
    z.zone_name,
    ROUND(AVG(r.safety_score), 1) AS avg_safety,
    COUNT(CASE WHEN r.safety_score < 60 THEN 1 END) AS risky_days,
    COUNT(*) AS total_days
FROM zones z
JOIN risk_scores r ON z.zone_id = r.zone_id
GROUP BY z.zone_name
ORDER BY risky_days DESC;
"""

risky_days = pd.read_sql(query2, conn)
risky_days.to_csv("risky_days.csv", index=False)
print(risky_days)

        zone_name  avg_safety  risky_days  total_days
0      Shamshabad        62.1          38          90
1        Kondapur        64.3          21          90
2   Jubilee Hills        63.8          20          90
3    Dilsukhnagar        66.9          15          90
4        Madhapur        70.7           6          90
5        Nacharam        73.9           5          90
6        Malakpet        70.7           5          90
7    Secunderabad        74.8           3          90
8       Manikonda        74.6           2          90
9         Attapur        74.1           2          90
10        Miyapur        81.6           1          90
11       LB Nagar        73.3           1          90
12     Kukatpally        79.3           1          90
13     Gachibowli        78.6           1          90
14      Charminar        80.9           1          90
15          Uppal        84.0           0          90
16    Hitech City        82.9           0          90
17       Begumpet        81.

In [26]:
# checking both csv files actually got saved
import os
print(os.listdir())

['.ipynb_checkpoints', 'Data_cleaning.ipynb', 'delivery_cost.csv', 'delivery_cost_clean.csv', 'desktop.ini', 'drone_delivery_clean.db', 'queries.sql', 'risky_days.csv', 'risk_scores_clean.csv', 'rollout_priority.csv', 'weather_clean.csv', 'weather_logs.csv', 'zones.csv', 'zones_clean.csv']


In [27]:
# reconnecting since we closed the connection earlier
conn = sqlite3.connect("drone_delivery_clean.db")

# running the safety consistency query and saving as csv
query3 = """
SELECT 
    z.zone_name,
    ROUND(AVG(r.safety_score), 1) AS avg_safety,
    ROUND(MIN(r.safety_score), 1) AS worst_day,
    ROUND(MAX(r.safety_score), 1) AS best_day
FROM zones z
JOIN risk_scores r ON z.zone_id = r.zone_id
GROUP BY z.zone_name
ORDER BY avg_safety DESC;
"""

safety_consistency = pd.read_sql(query3, conn)
safety_consistency.to_csv("safety_consistency.csv", index=False)
print(safety_consistency)

conn.close()

        zone_name  avg_safety  worst_day  best_day
0           Uppal        84.0       67.0      92.0
1        Ameerpet        83.7       65.4      93.9
2     Hitech City        82.9       65.3      93.7
3         Miyapur        81.6       51.0      90.0
4        Begumpet        81.5       66.7      90.6
5       Charminar        80.9       53.7      89.8
6   Banjara Hills        80.9       65.7      90.8
7      Kukatpally        79.3       52.5      90.4
8      Gachibowli        78.6       51.5      86.4
9    Secunderabad        74.8       53.4      84.5
10      Manikonda        74.6       53.5      84.5
11        Attapur        74.1       58.5      84.5
12       Nacharam        73.9       45.4      84.2
13       LB Nagar        73.3       58.5      83.6
14       Malakpet        70.7       45.5      79.9
15       Madhapur        70.7       49.9      81.4
16   Dilsukhnagar        66.9       55.5      76.2
17       Kondapur        64.3       47.4      75.0
18  Jubilee Hills        63.8  

In [28]:
# confirming all 3 dashboard-ready csv files are actually there
import os
print(os.listdir())

['.ipynb_checkpoints', 'Data_cleaning.ipynb', 'delivery_cost.csv', 'delivery_cost_clean.csv', 'desktop.ini', 'drone_delivery_clean.db', 'queries.sql', 'risky_days.csv', 'risk_scores_clean.csv', 'rollout_priority.csv', 'safety_consistency.csv', 'weather_clean.csv', 'weather_logs.csv', 'zones.csv', 'zones_clean.csv']
